# nb47 - Quantile mixture pooling of the nb44 ensemble (H11a)

**Error analysis.** nb44 pools seeds by AVERAGING their quantile predictions before calibration (0.0425 record). Averaging quantiles is not the statistically correct way to combine predictive distributions - the mixture of the five per-seed distributions has its own median and quartiles, and for skewed or disagreeing seeds the mixture median differs from the mean of medians.

**Question.** Does pooling the five seeds as a proper mixture distribution (average the CDFs, invert) beat quantile averaging?

**Hypothesis.** H11a: mixture-CDF pooling (and/or precision weighting by predicted width) improves the ensemble beyond 0.0425.

**Research.** Deep Gaussian Mixture Ensembles (arXiv:2306.07235); Bayesian deep-ensemble aggregation (2607.06776); the distributional-regression review flagged mean-of-means as the weak point and raw 1/sigma^2 weighting as unproven - both tested here.

**Proof criterion.** Inference-only on the fixed nb44 checkpoints and splits; win = >0.002 below 0.0425 overall or in any E>17 bin; smaller = noise and quantile averaging stays.

In [1]:
import os, sys, time, pathlib
import numpy as np, pandas as pd
import torch
REPO = pathlib.Path(os.environ['REPO_DIR']) if os.environ.get('REPO_DIR') else (
    pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())
sys.path.insert(0, str(REPO / 'scripts'))
from run_experiments import resolution
from picocal_data import build_grid, prep
from picocal_models import SubNetFQ, width_binned_calibration
OUT = REPO / 'reports' / 'predictions'
CKPT = REPO / '.scratch' / 'ckpt'
DEVICE = os.environ.get('NB47_DEVICE') or ('cuda' if torch.cuda.is_available() else 'cpu')
MODE = os.environ.get('NB47_MODE', 'full')
MBF = sorted((REPO / 'data' / 'minimum_bias').glob('*.root'))
CLF = sorted((REPO / 'data' / 'full').glob('matched_*.root'))
if MODE == 'smoke': MBF, CLF = MBF[:8], CLF[:4]
t0 = time.time()
ME = build_grid(MBF, 'minbias')
CE = build_grid(CLF, 'clean')
D = prep(4, ME, CE, ng=6)
T = dict(X=torch.from_numpy(D['X']).to(DEVICE), M=torch.from_numpy(D['M']).to(DEVICE),
         G=torch.from_numpy(D['G']).to(DEVICE), E=torch.from_numpy(D['Eraw']).to(DEVICE))
print(f'device {DEVICE} | mode {MODE} | build+prep {time.time()-t0:.0f}s')

minbias: 72554 events


clean: 30303 events


W=4: N 102857 (main 72554 + aux 30303), tr/va/te 50787/10883/10884, IN_DIM 16
device cuda | mode full | build+prep 140s


In [2]:
SEEDS = [0, 1, 2, 3, 4]
MODELS = []
for s in SEEDS:
    ck = CKPT / f'nb44_quantaux_s{s}.pt'
    if not ck.exists():
        print('missing', ck.name); continue
    mdl = SubNetFQ(D['IN_DIM'], D['la0'], D['lb0'], ng=6).to(DEVICE)
    mdl.load_state_dict(torch.load(ck, map_location=DEVICE)['bstate']); mdl.eval()
    MODELS.append(mdl)
print('loaded', len(MODELS), 'nb44 checkpoints')
def infer(idx):
    per = []
    with torch.no_grad():
        for mdl in MODELS:
            out = []
            for j in range(0, len(idx), 256):
                b = torch.from_numpy(np.asarray(idx[j:j+256])).to(DEVICE)
                out.append(mdl(T['X'][b], T['M'][b], T['G'][b], T['E'][b]).cpu().numpy())
            per.append(np.concatenate(out))
    return np.stack(per)
kva, kte = D['kva'], D['kte']
QV = infer(kva); QT = infer(kte)
print('QV', QV.shape, 'QT', QT.shape)

loaded 5 nb44 checkpoints


QV (5, 10883, 3) QT (5, 10884, 3)


Mixture pooling: each seed's 3 quantiles define a piecewise-linear CDF (linear tails extrapolated from the adjacent quartile segment); the ensemble CDF is their average; pooled quartiles are read off the averaged CDF on a per-event grid.

In [3]:
def mix_quantiles(Q, npts=241):
    S, N, _ = Q.shape
    lo = Q[:, :, 0].min(0) - 2 * (Q[:, :, 1] - Q[:, :, 0]).max(0)
    hi = Q[:, :, 2].max(0) + 2 * (Q[:, :, 2] - Q[:, :, 1]).max(0)
    grid = lo[:, None] + (hi - lo)[:, None] * np.linspace(0, 1, npts)[None, :]
    cdf = np.zeros((N, npts))
    for s in range(S):
        q25, q50, q75 = Q[s, :, 0][:, None], Q[s, :, 1][:, None], Q[s, :, 2][:, None]
        g = grid
        s1 = 0.25 / np.maximum(q50 - q25, 1e-6)
        s2 = 0.25 / np.maximum(q75 - q50, 1e-6)
        c = np.where(g < q50, 0.5 + (g - q50) * s1, 0.5 + (g - q50) * s2)
        cdf += np.clip(c, 0.0, 1.0)
    cdf /= S
    out = np.zeros((N, 3))
    for k, p in enumerate((0.25, 0.5, 0.75)):
        idx = np.argmax(cdf >= p, axis=1)
        i0 = np.clip(idx - 1, 0, npts - 1)
        c0 = cdf[np.arange(N), i0]; c1 = cdf[np.arange(N), idx]
        g0 = grid[np.arange(N), i0]; g1 = grid[np.arange(N), idx]
        w = np.where(c1 > c0, (p - c0) / np.maximum(c1 - c0, 1e-9), 0.0)
        out[:, k] = g0 + w * (g1 - g0)
    return out
yva = D['y'][kva]; Et_te = D['Et'][kte]
qv_mean, qt_mean = QV.mean(0), QT.mean(0)
pe_mean = width_binned_calibration(qv_mean, qt_mean, yva)
qv_mix, qt_mix = mix_quantiles(QV), mix_quantiles(QT)
pe_mix = width_binned_calibration(qv_mix, qt_mix, yva)
wv = 1.0 / np.maximum(QV[:, :, 2] - QV[:, :, 0], 1e-6) ** 2
wt = 1.0 / np.maximum(QT[:, :, 2] - QT[:, :, 0], 1e-6) ** 2
qv_prec = (QV * wv[:, :, None]).sum(0) / wv.sum(0)[:, None]
qt_prec = (QT * wt[:, :, None]).sum(0) / wt.sum(0)[:, None]
pe_prec = width_binned_calibration(qv_prec, qt_prec, yva)
for name, pe in (('quantile mean (nb44 method)', pe_mean), ('mixture CDF', pe_mix), ('precision-weighted', pe_prec)):
    print(f'{name:28s}: {resolution(pe, Et_te)["sigma_eff"]:.4f}')

quantile mean (nb44 method) : 0.0425
mixture CDF                 : 0.0425
precision-weighted          : 0.0425


## Verdict vs the 0.0425 record

Per-bin for each pooling; win = >0.002 anywhere that matters, else quantile averaging stands.

In [4]:
edges = np.quantile(Et_te, np.linspace(0, 1, 7))
def perbin(pe):
    out = []
    for i in range(6):
        hi = edges[i+1] + (1e-9 if i == 5 else 0)
        mm = (Et_te >= edges[i]) & (Et_te < hi)
        out.append(resolution(pe[mm], Et_te[mm])['sigma_eff'])
    return out
print('record nb44 (TTA) per-bin: 0.0657/0.0483/0.0364/0.0361/0.0345/0.0354 | targets 0.06/0.045/0.035/0.032/0.030/0.030')
for name, pe in (('mean', pe_mean), ('mixture', pe_mix), ('precision', pe_prec)):
    print(f'{name:10s} overall {resolution(pe, Et_te)["sigma_eff"]:.4f} | per-bin ' + ' / '.join(f'{b:.4f}' for b in perbin(pe)))
np.save(OUT / f'nb47_pred_mixture.npy', pe_mix)

record nb44 (TTA) per-bin: 0.0657/0.0483/0.0364/0.0361/0.0345/0.0354 | targets 0.06/0.045/0.035/0.032/0.030/0.030
mean       overall 0.0425 | per-bin 0.0665 / 0.0481 / 0.0363 / 0.0360 / 0.0348 / 0.0359
mixture    overall 0.0425 | per-bin 0.0665 / 0.0479 / 0.0360 / 0.0362 / 0.0348 / 0.0357
precision  overall 0.0425 | per-bin 0.0660 / 0.0478 / 0.0361 / 0.0360 / 0.0349 / 0.0356


## Act 3a - sigma_eff-direct calibration (spec section 2)

The final linear calibration is fit by least squares, which optimizes MSE - not our quantile-core metric. Here the same 2 parameters per width group are fit by directly minimizing sigma_eff on the validation set (precedent: Belle II selects by FWHM on validation, arXiv:2306.04179; ATLAS fits scale/smearing on holdout, arXiv:2309.05471). Criterion: adopt if val and test move in the same direction; test reported once.

In [5]:
from scipy.optimize import minimize
Ev_va = np.exp(yva)
def direct_wcalib(qv, qt, yva_, Ev):
    wv = qv[:, 2] - qv[:, 0]; wt_ = qt[:, 2] - qt[:, 0]
    cuts = np.quantile(wv, [1/3, 2/3])
    gv = np.digitize(wv, cuts); gt = np.digitize(wt_, cuts)
    p0 = []
    for g in range(3):
        a0, b0 = np.polyfit(qv[gv == g, 1], yva_[gv == g], 1)
        p0 += [a0, b0]
    def apply(p, q, grp):
        pe = np.empty(len(q))
        for g in range(3):
            pe[grp == g] = np.exp(p[2*g] * q[grp == g, 1] + p[2*g+1])
        return pe
    def obj(p):
        return resolution(apply(p, qv, gv), Ev)['sigma_eff']
    res = minimize(obj, p0, method='Nelder-Mead',
                   options=dict(xatol=1e-5, fatol=1e-7, maxiter=3000))
    p = res.x if res.fun <= obj(np.array(p0)) else np.array(p0)
    return apply(p, qv, gv), apply(p, qt, gt)
pe_v_dir, pe_t_dir = direct_wcalib(qv_mean, qt_mean, yva, Ev_va)
pe_v_ls = width_binned_calibration(qv_mean, qv_mean, yva)
print(f'val : polyfit {resolution(pe_v_ls, Ev_va)["sigma_eff"]:.4f} -> direct {resolution(pe_v_dir, Ev_va)["sigma_eff"]:.4f}')
print(f'test: polyfit {resolution(pe_mean, Et_te)["sigma_eff"]:.4f} -> direct {resolution(pe_t_dir, Et_te)["sigma_eff"]:.4f}')
np.save(OUT / 'nb47_pred_directcal.npy', pe_t_dir)

val : polyfit 0.0411 -> direct 0.0402
test: polyfit 0.0425 -> direct 0.0419
